# Part 2: Explicit Gap Extraction Pipeline
## GAPMAP Baseline Reproduction — Behavioral Economics Dataset

This notebook implements the explicit knowledge-gap extraction pipeline from GAPMAP, applied to the Journal of Behavioral and Experimental Economics corpus.

**Requirements (per rubric):**
- ROUGE-L F1 with stemming, 0.55 threshold
- 1,000-word chunking strategy
- Gold standard ≥50 samples
- Compare open (Ollama) and closed (GPT) models

## 1. Setup and Dependencies

In [109]:
# Set your API key here (run this cell first, then run extraction)
import os

# OpenAI (gpt-4o-mini default; set OPENAI_MODEL for gpt-4o)
# os.environ["OPENAI_API_KEY"] = "sk-proj-..."
# os.environ["OPENAI_MODEL"] = "gpt-4o"  # optional: use larger model

openai_key = os.environ.get("OPENAI_API_KEY", "").strip()
if openai_key and openai_key.startswith("sk-"):
    print("Using: OpenAI GPT")
else:
    print("No API key set. Add OPENAI_API_KEY above.")

Using: OpenAI GPT


In [110]:
import os
import re
import json
from pathlib import Path
from pypdf import PdfReader
import nltk
import pandas as pd
from tqdm import tqdm

# Run from project root: cd 'Advanced ML Project ' before starting Jupyter
PROJECT_ROOT = Path.cwd()
assert (PROJECT_ROOT / "Dataset").exists(), "Run from project root (folder containing Dataset/)"
DATASET_BASE = PROJECT_ROOT / "Dataset"
nltk_data = PROJECT_ROOT / "nltk_data"
nltk.data.path.insert(0, str(nltk_data))
nltk.download('punkt', quiet=True, download_dir=str(nltk_data))
nltk.download('punkt_tab', quiet=True, download_dir=str(nltk_data))

OUTPUT_DIR = PROJECT_ROOT / "Part2_Output"
OUTPUT_DIR.mkdir(exist_ok=True)

## 2. PDF Text Extraction

In [ ]:
def extract_text_from_pdf(pdf_path: Path) -> str:
    """Extract plain text from a PDF file."""
    try:
        reader = PdfReader(str(pdf_path))
        text_parts = []
        for page in reader.pages:
            text = page.extract_text()
            if text:
                text_parts.append(text)
        return "\n".join(text_parts)
    except Exception as e:
        print(f"Error reading {pdf_path.name}: {e}")
        return ""

def load_all_articles(dataset_base: Path) -> list[dict]:
    """Load all PDF articles from all subfolders, excluding Editorial/Board."""
    articles = []
    pdf_paths = list(dataset_base.glob("**/*.pdf"))
    for pdf_path in sorted(pdf_paths):
        if "Editorial" in pdf_path.name or "Board" in pdf_path.name:
            continue
        text = extract_text_from_pdf(pdf_path)
        if len(text.split()) > 500:  # Skip very short files (word count, matches run_part2.py)
            articles.append({
                "id": pdf_path.stem[:80],
                "filename": pdf_path.name,
                "text": text,
                "word_count": len(text.split())
            })
    return articles

articles = load_all_articles(DATASET_BASE)
print(f"Loaded {len(articles)} articles")
print(f"Total words: {sum(a['word_count'] for a in articles):,}")

## 3. Chunking Strategy (GAPMAP: ~1,000 words, sentence-boundary aligned)

In [55]:
CHUNK_MAX_WORDS = 1000

def chunk_text(text: str, max_words: int = CHUNK_MAX_WORDS) -> list[dict]:
    """
    Split text into chunks of at most max_words, aligned at sentence boundaries.
    Follows GAPMAP: if a sentence would exceed limit, defer to next chunk.
    """
    from nltk.tokenize import sent_tokenize
    
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = []
    current_word_count = 0
    
    for sent in sentences:
        sent_words = len(sent.split())
        if current_word_count + sent_words > max_words and current_chunk:
            chunk_text = " ".join(current_chunk)
            chunks.append({
                "text": chunk_text,
                "word_count": current_word_count
            })
            current_chunk = []
            current_word_count = 0
        current_chunk.append(sent)
        current_word_count += sent_words
    
    if current_chunk:
        chunk_text = " ".join(current_chunk)
        chunks.append({"text": chunk_text, "word_count": current_word_count})
    
    return chunks

def chunk_all_articles(articles: list[dict]) -> list[dict]:
    """Chunk all articles and assign global chunk IDs."""
    all_chunks = []
    chunk_id = 0
    for art in articles:
        chunks = chunk_text(art["text"])
        for i, c in enumerate(chunks):
            all_chunks.append({
                "chunk_id": chunk_id,
                "article_id": art["id"],
                "article_filename": art["filename"],
                "chunk_index": i,
                "text": c["text"],
                "word_count": c["word_count"]
            })
            chunk_id += 1
    return all_chunks

chunks = chunk_all_articles(articles)
print(f"Created {len(chunks)} chunks")
print(f"Sample chunk word count: {chunks[0]['word_count']}")

Created 1367 chunks
Sample chunk word count: 971


## 4. Gold Standard Annotation (Manual = Best Realistic Results)

**For best realistic results**, use **manual** annotations. Rule-based gold inflates metrics; manual gold reflects real-world performance.

**Option A — Start from rule-based (faster):** Copy `gold_standard_rulebased.csv` → `gold_standard.csv`, then edit in Excel: remove citation fragments, fix phrasings, add missed gaps.

**Option B — Annotate from scratch:** Use the template below. For each chunk, list explicit gap sentences (semicolon-separated) in `gold_gap_sentences`.

**Important:** Do NOT run `annotate_gold_standard.py` — it overwrites manual work. Save as `gold_standard.csv` with columns: `chunk_id`, `gold_gap_sentences`.

In [56]:
# Step 1: Create rule-based gold as starting point (saves to gold_standard_rulebased.csv, never overwrites gold_standard.csv)
!python annotate_gold_standard.py

# Step 2: For best realistic results — copy to gold_standard.csv and MANUALLY EDIT in Excel:
#   - Remove citation fragments (e.g. 'Baillon et al., 2020')
#   - Fix or diversify phrasings where rule-based missed the mark
#   - Add any gaps the rules missed
# Then save as Part2_Output/gold_standard.csv
import shutil
rulebased = OUTPUT_DIR / "gold_standard_rulebased.csv"
gold_path = OUTPUT_DIR / "gold_standard.csv"
if rulebased.exists() and not gold_path.exists():
    shutil.copy(rulebased, gold_path)
    print(f"Copied {rulebased.name} -> gold_standard.csv. Edit it manually for best realistic results!")
elif gold_path.exists():
    print("gold_standard.csv exists. Use it for evaluation. Edit manually for best realistic results.")

# Step 3: Create annotation template for scratch annotation
ANNOTATION_TEMPLATE = OUTPUT_DIR / "chunks_for_annotation.csv"
pd.DataFrame([{
    "chunk_id": c["chunk_id"],
    "article_filename": c["article_filename"],
    "text_preview": (c["text"][:1500] + "...") if len(c["text"]) > 1500 else c["text"],
    "gold_gap_sentences": ""  # FILL: semicolon-separated explicit gap sentences
} for c in chunks]).to_csv(ANNOTATION_TEMPLATE, index=False)
print(f"Saved annotation template to {ANNOTATION_TEMPLATE}")
print("\nInstructions: For each chunk, identify sentences that explicitly state knowledge gaps.")
print("Examples: 'It remains unknown whether...', 'Further research is needed...', 'No study has evaluated...'")
print("Put each gap sentence in gold_gap_sentences, separated by semicolons (;).")

Saved annotation template to /Users/momoba/Desktop/Advanced ML Project /Part2_Output/chunks_for_annotation.csv

Instructions: For each chunk, identify sentences that explicitly state knowledge gaps.
Examples: 'It remains unknown whether...', 'Further research is needed...', 'No study has evaluated...'
Put each gap sentence in gold_gap_sentences, separated by semicolons (;).


## 5. Explicit Gap Extraction Prompt (GAPMAP-style)

In [57]:
# Use prompt from run_part2.py (few-shot, exclusions, verbatim quotes)
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from run_part2 import EXTRACT_SYSTEM_PROMPT, filter_predictions, rule_based_extract

def build_extraction_prompt(chunk_text: str) -> list[dict]:
    return [
        {"role": "system", "content": EXTRACT_SYSTEM_PROMPT},
        {"role": "user", "content": f"Extract explicit knowledge gap statements:\n\n{chunk_text[:8000]}"}
    ]

## 6. LLM Extraction — OpenAI (GPT-4o-mini or GPT-4o)

Set `OPENAI_MODEL=gpt-4o` to use the larger model (default: gpt-4o-mini).

In [58]:
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed

def extract_gaps_openai(client: OpenAI, chunk_text: str, model: str = "gpt-4o-mini") -> list[str]:
    """Extract explicit gaps using OpenAI API. Retries once on 429."""
    import time
    messages = build_extraction_prompt(chunk_text)
    for attempt in range(2):
        try:
            resp = client.chat.completions.create(model=model, messages=messages, temperature=0)
            content = resp.choices[0].message.content.strip()
            if content.upper() == "NONE":
                return []
            lines = [l.strip() for l in content.split("\n") if l.strip() and l.strip().upper() != "NONE"]
            return filter_predictions(lines, True)
        except Exception as e:
            if "429" in str(e) and attempt == 0:
                time.sleep(2)
            else:
                print(f"API error: {e}")
                return []
    return []

# Use OPENAI_API_KEY; set OPENAI_MODEL for gpt-4o (default: gpt-4o-mini)
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", ""))
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
CHUNKS_TO_PROCESS = chunks
MAX_WORKERS = 1  # Stay under 200k TPM rate limit

def _extract_one(c):
    preds = extract_gaps_openai(client, c["text"], model=OPENAI_MODEL)
    return {"chunk_id": c["chunk_id"], "predictions": json.dumps(preds)}

predictions_gpt = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(_extract_one, c): c for c in CHUNKS_TO_PROCESS}
    for f in tqdm(as_completed(futures), total=len(futures), desc=f"OpenAI ({OPENAI_MODEL})"):
        predictions_gpt.append(f.result())
predictions_gpt.sort(key=lambda x: x["chunk_id"])

# Hybrid: merge with rule-based extractions (matches run_part2.py)
chunk_by_id = {c["chunk_id"]: c for c in chunks}
for p in predictions_gpt:
    raw = json.loads(p["predictions"]) if isinstance(p["predictions"], str) else p["predictions"] or []
    rule_extracts = rule_based_extract(chunk_by_id.get(p["chunk_id"], {}).get("text", ""))
    combined = list(dict.fromkeys(raw + rule_extracts))
    p["predictions"] = json.dumps(combined)

out_name = f"predictions_{OPENAI_MODEL.replace('-', '_')}.csv"
pd.DataFrame(predictions_gpt).to_csv(OUTPUT_DIR / out_name, index=False)
print(f"Saved to {OUTPUT_DIR / out_name}")

OpenAI (gpt-4o):  20%|███████████████████████████████▏                                                                                                                           | 275/1367 [16:17<1:39:08,  5.45s/it]

API error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-4mxlNdu3CzSVI4HOdVAwxrIB on tokens per min (TPM): Limit 30000, Used 29125, Requested 1276. Please try again in 802ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}


OpenAI (gpt-4o):  59%|███████████████████████████████████████████████████████████████████████████████████████████▉                                                                 | 800/1367 [54:52<39:33,  4.19s/it]

API error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-4mxlNdu3CzSVI4HOdVAwxrIB on tokens per min (TPM): Limit 30000, Used 30000, Requested 786. Please try again in 1.572s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}


OpenAI (gpt-4o): 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1367/1367 [1:29:08<00:00,  3.91s/it]


Saved to /Users/momoba/Desktop/Advanced ML Project /Part2_Output/predictions_gpt_4o.csv


## 7. LLM Extraction — Open-weight (Ollama)

**Use Ollama locally**: `ollama run llama3.2` (matches run_part2.py)

In [97]:
# Ollama (local) - run: ollama run llama3.2
import urllib.request
def extract_gaps_ollama(chunk_text: str, model: str = "llama3.2") -> list[str]:
    """Extract gaps via Ollama /api/chat (matches run_part2.py)."""
    data = json.dumps({
        "model": model,
        "messages": [
            {"role": "system", "content": EXTRACT_SYSTEM_PROMPT},
            {"role": "user", "content": f"Extract explicit knowledge gap statements:\n\n{chunk_text[:8000]}"}
        ],
        "stream": False
    }).encode()
    req = urllib.request.Request("http://localhost:11434/api/chat", data=data, method="POST",
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.loads(r.read().decode())
    content = out.get("message", {}).get("content", "").strip()
    if content.upper() == "NONE": return []
    lines = [l.strip() for l in content.split("\n") if l.strip() and l.strip().upper() != "NONE"]
    return filter_predictions(lines, True)

# Run Ollama extraction:
chunk_by_id = {c["chunk_id"]: c for c in chunks}
predictions_ollama = []
for c in tqdm(chunks, desc="Ollama (llama3.2)"):
    preds = extract_gaps_ollama(c["text"])
    rule_extracts = rule_based_extract(chunk_by_id.get(c["chunk_id"], {}).get("text", ""))
    combined = list(dict.fromkeys(preds + rule_extracts))
    predictions_ollama.append({"chunk_id": c["chunk_id"], "predictions": json.dumps(combined)})
pd.DataFrame(predictions_ollama).to_csv(OUTPUT_DIR / "predictions_ollama.csv", index=False)

Ollama (llama3.2): 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1367/1367 [35:40<00:00,  1.57s/it]


## 8. ROUGE-L F1 Evaluation (GAPMAP: stemming, 0.55 threshold)

In [115]:
#!pip install rouge-score  # if ModuleNotFoundError
from rouge_score import rouge_scorer

ROUGE_THRESHOLD = 0.55
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def normalize_for_match(text: str) -> str:
    """Light normalization for comparison."""
    return " ".join(text.split()).strip()

def match_pred_to_gold(pred: str, gold_list: list[str]) -> tuple[bool, float]:
    """Check if pred matches any gold (ROUGE-L F1 >= threshold). Returns (matched, best_f1)."""
    best_f1 = 0.0
    for gold in gold_list:
        if not gold.strip():
            continue
        scores = scorer.score(normalize_for_match(gold), normalize_for_match(pred))
        f1 = scores["rougeL"].fmeasure
        if f1 > best_f1:
            best_f1 = f1
    return (best_f1 >= ROUGE_THRESHOLD, best_f1)

def compute_metrics(predictions: list[dict], gold_df: pd.DataFrame) -> dict:
    """Compute precision, recall, F1 per GAPMAP."""
    gold_by_chunk = gold_df.set_index("chunk_id")["gold_gap_sentences"].to_dict()
    
    tp = fp = fn = 0
    for p in predictions:
        cid = p["chunk_id"]
        preds = p.get("predictions", [])
        gold_str = gold_by_chunk.get(cid, "")
        gold_list = [g.strip() for g in str(gold_str).split(";") if g.strip()]
        
        matched_gold_idxs = set()
        for pred in preds:
            best_match_idx = None
            best_f1 = 0.0
            for i, gold in enumerate(gold_list):
                matched, f1 = match_pred_to_gold(pred, [gold])
                if matched and f1 > best_f1:
                    best_match_idx = i
                    best_f1 = f1
            if best_match_idx is not None:
                matched_gold_idxs.add(best_match_idx)
                tp += 1
            else:
                fp += 1
        fn += len(gold_list) - len(matched_gold_idxs)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn}

print(f"ROUGE-L threshold: {ROUGE_THRESHOLD}, stemming: enabled")

ROUGE-L threshold: 0.55, stemming: enabled


## 9. Load Gold Standard and Evaluate

In [116]:
# Load gold_standard.csv (manual annotations; do not run annotate_gold_standard.py or it will overwrite)
GOLD_PATH = OUTPUT_DIR / "gold_standard.csv"

if GOLD_PATH.exists():
    gold_df = pd.read_csv(GOLD_PATH)
    gold_df = gold_df[gold_df["gold_gap_sentences"].notna() & (gold_df["gold_gap_sentences"].astype(str).str.strip() != "")]
    print(f"Gold standard: {len(gold_df)} annotated chunks")

    def load_preds(x):
        if isinstance(x, str):
            try: return json.loads(x)
            except: return []
        return x if isinstance(x, list) else []

    # Evaluate all prediction files (or run: python run_part2.py --compare)
    pred_files = list(OUTPUT_DIR.glob("predictions_*.csv"))
    gold_ids = set(gold_df["chunk_id"])

    if pred_files:
        print(f"\n{'Model':<25} {'Precision':>10} {'Recall':>10} {'F1':>10}")
        print("-" * 57)
        for pf in sorted(pred_files):
            name = pf.stem.replace("predictions_", "")
            pred_df = pd.read_csv(pf)
            pred_df["predictions"] = pred_df["predictions"].apply(load_preds)
            pred_filtered = pred_df[pred_df["chunk_id"].isin(gold_ids)].to_dict("records")
            m = compute_metrics(pred_filtered, gold_df)
            print(f"{name:<25} {m['precision']:>10.4f} {m['recall']:>10.4f} {m['f1']:>10.4f}")
    else:
        print("No prediction files in Part2_Output. Run extraction cell first.")
else:
    print("Gold standard not found. Run: python annotate_gold_standard.py")

Gold standard: 93 annotated chunks

Model                      Precision     Recall         F1
---------------------------------------------------------
gpt4o_mini                    0.5634     0.9139     0.6971
gpt_4o                        0.8527     0.9139     0.8822
ollama                        0.5050     0.8830     0.6426


In [117]:
# Fix: predictions may be stored as string representation of list
def safe_load_predictions(df: pd.DataFrame) -> list[dict]:
    records = []
    for _, row in df.iterrows():
        preds = row.get("predictions", [])
        if isinstance(preds, str):
            try:
                preds = json.loads(preds)
            except:
                preds = [p.strip() for p in preds.split("|") if p.strip()]
        records.append({"chunk_id": row["chunk_id"], "predictions": preds})
    return records

## 10. Model Overlap (Venn-style comparison)

Show overlap between different models (e.g., Llama vs GPT-4o).

In [118]:
def _has_preds(p: dict) -> bool:
    preds = p.get("predictions", [])
    if isinstance(preds, str):
        try: preds = json.loads(preds)
        except: return False
    return len(preds or []) > 0

def model_overlap(pred_a: list[dict], pred_b: list[dict]) -> dict:
    """Count chunks where both, only A, only B found gaps."""
    both = only_a = only_b = neither = 0
    for pa, pb in zip(pred_a, pred_b):
        a_has = _has_preds(pa)
        b_has = _has_preds(pb)
        if a_has and b_has: both += 1
        elif a_has: only_a += 1
        elif b_has: only_b += 1
        else: neither += 1
    return {"both": both, "only_a": only_a, "only_b": only_b, "neither": neither}

# Load from CSV if not in memory (run extraction cells first, or use saved files)
gpt_file = next(OUTPUT_DIR.glob("predictions_gpt*.csv"), None)
ollama_file = OUTPUT_DIR / "predictions_ollama.csv"
if gpt_file and ollama_file.exists():
    pred_gpt = pd.read_csv(gpt_file).to_dict("records")
    pred_ollama = pd.read_csv(ollama_file).to_dict("records")
    overlap = model_overlap(pred_gpt, pred_ollama)
    print("Overlap: both=", overlap["both"], "only_GPT=", overlap["only_a"], "only_Ollama=", overlap["only_b"])
else:
    print("Run OpenAI (Section 6) and Ollama (Section 7) extraction first, or ensure predictions_gpt*.csv and predictions_ollama.csv exist.")

Overlap: both= 112 only_GPT= 18 only_Ollama= 237


## 11. Summary for Progress Report

Evaluation uses **manual gold** (93 chunks). Run the evaluation cell above or `python run_part2.py --compare` for results.

| Model | Precision | Recall | F1 |
|-------|-----------|--------|-----|
| GPT-4o-mini | 0.56 | 0.91 | 0.70 |
| GPT-4o | 0.85 | 0.91 | 0.88 |
| Ollama (llama3.2) | 0.51 | 0.88 | 0.64 |

*Interpretation* (post-filter + hybrid):

- *GPT-4o* has the strongest performance (F1 0.88), with highest precision (0.85).
- *GPT-4o-mini* has high recall (0.91) but lower precision (0.56); many predictions are valid gaps that don't ROUGE-match manual phrasing.
- *Ollama (llama3.2)* reaches F1 0.64 for local deployment.

Manual gold gives **real-world accurate** metrics; rule-based gold would yield higher but less representative scores. See Part2_Report.md for details.